# Can editability be INDUCED BY TRAINING?

**Direction:** `research/directions/trained-editability.md` · `[in-frame]` · sub-Q 3 (editability). **Model:** GRU only.
**Branch:** `more_trained_editability`.

**The gap.** Every §4 result so far uses **inference-time** editors on a **frozen** world model, and every
probe-directed one fails: on the same model the decoder-gradient oracle reaches the edited world (Edit Index
**+0.94**) while readout injection, Global-PCA projection and PCA geodesic all sit at the unedited end (≈ −0.6).
`learn_to_edit` asked whether *training* fixes this and returned two negatives — but both under a deliberately
**light** budget, with a heavier fine-tune explicitly left **OWED** in `METRICS_AND_EDITORS.md`. This notebook pays
that debt and turns it into a controlled comparison.

**Two mechanisms, one evaluation.**
- **Fine-tune the world model** so that a **fixed, untrained** editor works. The editor is the linear-pseudoinverse
  *readout injection* whose probe is fit once on the base model and then **frozen** — nothing about the editor is
  learned, so all adaptation is in the world model, which must learn to honour writes along `A⁺` as "put the object
  here".
- **Amortized editor**: freeze the world model and learn `E_θ(h, target) → Δh` instead.

**The four questions.** (1) Does training make the edit work *at all*? (2) Does it **generalise across write
mechanisms** — does a model taught to obey one editor obey the others, or has it wired a single button?
(3) Does it **generalise across content** — train on object-0 teleports, test on object-1? (4) What does it
**cost** the world model?

All training uses `edits[2000:]`; **everything reported here is the held-out `edits[:64]`**, the same samples the
`controls/` notebooks report on, so numbers are directly comparable.

## Definitions

### Runs (canonical registry: `TRAINED_EDITABILITY_RUNS.md`)

All arms start from `runs/controls/H256` (GRU H=256, `datasets/4_fixed_refl_inview`, obs noise 0.2, position noise
0.04, 400 epochs) and are trained by `scripts/train_editable_gru.py`, batch 64, Adam lr 1e-4.

| code | descriptive label (used in every figure) | what is trained | steps | retention weight | training edits |
|---|---|---|---|---|---|
| `H256` | **base model (no editability training)** | nothing — the frozen baseline | — | — | — |
| `FT_light` | **fine-tuned · light (300 steps)** | the world model | 300 | 1.0 | both objects |
| `FT_heavy` | **fine-tuned · heavy (3000 steps)** | the world model | 3000 | 1.0 | both objects |
| `FT_heavy_noret` | **fine-tuned · heavy · no retention** | the world model | 3000 | **0.0** | both objects |
| `FT_heavy_obj0` | **fine-tuned · heavy · object-0 edits only** | the world model | 3000 | 1.0 | **object 0 only** |
| `AMORT` | **amortized editor (world model frozen)** | `E_θ(h,target)→Δh` only | 3000 | — | both objects |

**Losses.** `edit = MSE(rollout(h_edited, K=15), clean_obs[ef:ef+K])`; `retention` = ordinary teacher-forced
next-step MSE on test sequences; `total = edit + retention_weight · retention`. The retention term is what
separates *"the model became editable"* from *"the model was destroyed and now renders whatever it is asked for"*.

### The three write mechanisms this notebook distinguishes — read this before the tables

| name | what it is | why it is here |
|---|---|---|
| **Trained interface** | readout injection through **the exact frozen probe the arm was fine-tuned for** | did the training work *at all*? |
| **Readout injection (fresh probe)** | the same *mechanism*, but with a linear probe re-fit on the **fine-tuned** model's own states | **mechanism generalisation**: an equivalent write the model was not literally trained on |
| Global-PCA projection · PCA geodesic · MLP-probe gradient | the other standard §4 editors, untouched by training | do *other* write mechanisms benefit? |
| Decoder gradient (**oracle**) | Adam on `h` to match the true edit-frame observation | the bracket — is a target-rendering state still reachable at all? |
| Oracle observation (**reference**) | teacher-force one extra frame, the real **noisy** `edits.obs[ef]` — the model just *sees* the teleport | belief inertia; not an editor |

### Metrics (canonical §4 set — `../METRICS_AND_EDITORS.md` §4, implemented in `scripts/editability_metrics.py`)

| name | formula | units | better |
|---|---|---|---|
| **Edit Index** | `(d_uned − d_edit)/(d_uned + d_edit)`, `d_· = RMSE(edited₀, gt_·)` over the **differing rays** | −1…+1 | ↑ |
| **Edit Index by step** | the same at every rollout step, against the counterfactual world **rolled forward** | −1…+1 | ↑ |
| **Target / Ghost / Collateral / Edit-frame RMSE** | `RMSE(edited₀, gt_edited)` over target rays / ghost rays / the other object's rays / all rays | obs intensity | ↓ |
| **GT-traj RMSE** | `mean_s RMSE(edited_s, clean_obs[ef+s])` over the rollout | obs intensity | ↓ |
| **fidelity ratio** | `GT-traj RMSE(editor) / GT-traj RMSE(unsteered)` | ratio | ↓ (>1 = worse than doing nothing) |
| next-step RMSE, position/velocity R², fiber residual | as in `../controls/` — what the training **cost** | — | — |

> **How to read the Edit Index.** **+1** = the output *is* the world where the edit happened · **0** = equidistant
> from both (ambiguous, or garbage) · **−1** = it *is* the world where the edit did not happen. **Read it against
> that arm's own unsteered row**: a perfect predictor scores exactly −1 unsteered, a real one falls short by its own
> blur, so a *worse* predictor has a *higher* unsteered index (measured across 8 models: r = +0.987 with next-step
> RMSE). This matters here — a fine-tune that damages prediction moves the whole scale.

In [ ]:
# [1] Setup: load every arm, its frozen probe, the held-out edit set, and the canonical §4 machinery.
import os, sys, json
sys.path.insert(0, "../../../..")
sys.path.insert(0, "../../../../scripts")
import numpy as np, torch, h5py
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from IPython.display import display, Markdown

from pim.world_models import load_checkpoint, load_dataset
from pim.figures.theme import style_ax
from editability_metrics import build_edit_zones, edit_scorecard, fidelity_ratio
from train_editable_gru import AmortizedEditor, readout_inject

torch.manual_seed(0); np.random.seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_OBJ, K, N_EDIT = 2, 15, 64
OUT = "/tmp/trained_editability"; os.makedirs(OUT, exist_ok=True)

RUNS  = ["H256", "FT_light", "FT_heavy", "FT_heavy_noret", "FT_heavy_obj0", "AMORT"]
LABEL = {"H256": "base model\n(no editability training)",
         "FT_light": "fine-tuned\nlight (300 steps)",
         "FT_heavy": "fine-tuned\nheavy (3000 steps)",
         "FT_heavy_noret": "fine-tuned heavy\nNO retention",
         "FT_heavy_obj0": "fine-tuned heavy\nobject-0 edits only",
         "AMORT": "amortized editor\n(world model frozen)"}
SHORT = {k: v.replace("\n", " ") for k, v in LABEL.items()}
COLOR = {"H256": "0.45", "FT_light": "#56B4E9", "FT_heavy": "#0072B2",
         "FT_heavy_noret": "#D55E00", "FT_heavy_obj0": "#E69F00", "AMORT": "#009E73"}
ROOT  = {r: ("../../../../runs/controls" if r == "H256" else "../../../../runs/trained_editability") for r in RUNS}
EVALJ = {r: json.load(open(f"{ROOT[r]}/eval/{r}.json")) for r in RUNS}

# ── the held-out edit set (edits[:64] — never trained on) ─────────────────────
bundle = load_dataset("../../../../datasets/4_fixed_refl_inview", n_obj_keep=N_OBJ)
edits, test = bundle.edits, bundle.test
ef = edits.edit_frame; sim = test.config["dataset"]["sim"]
N = N_EDIT
oe = edits.edit_object[:N].astype(int)
with h5py.File(edits.h5_path, "r") as f:
    pre_vel = f["velocities"][:N, ef-1, :N_OBJ, :].astype(np.float32)
gt_roll = edits.clean_obs[:N, ef:ef+K, :].astype(np.float32)
tgt_pos = edits.positions[:N, ef, :N_OBJ, :].astype(np.float32)
target4 = torch.from_numpy(tgt_pos.reshape(N, N_OBJ*2)).float().to(DEVICE)
obs_t   = torch.from_numpy(edits.obs[:N]).float().to(DEVICE)
ZONES = build_edit_zones(pre_pos=edits.positions[:N, ef-1, :N_OBJ, :].astype(np.float32),
                         tgt_pos=tgt_pos, pre_vel=pre_vel, edit_object=oe, sim=sim, n_obj=N_OBJ,
                         traj_pos=edits.positions[:N, ef:ef+K, :N_OBJ, :].astype(np.float32),
                         gt_edited_traj=gt_roll)

# ── models + the frozen probe each arm was trained for ───────────────────────
MODELS, PROBE, EDITORNET = {}, {}, {}
for r in RUNS:
    MODELS[r], _ = load_checkpoint(f"{ROOT[r]}/{r}/best_model.pt", device=DEVICE)
    pf = f"{ROOT[r]}/{r}/frozen_probe.npz"
    if os.path.exists(pf):
        z = np.load(pf)
        PROBE[r] = tuple(torch.tensor(z[k], dtype=torch.float32, device=DEVICE) for k in ("W", "b", "W_pinv"))
    ef_path = f"{ROOT[r]}/{r}/amortized_editor.pt"
    if os.path.exists(ef_path):
        ck = torch.load(ef_path, map_location=DEVICE, weights_only=False)
        net = AmortizedEditor(ck["hidden"]).to(DEVICE); net.load_state_dict(ck["editor_state"]); net.eval()
        EDITORNET[r] = net
# the base model has no frozen probe of its own; it shares FT_heavy's so the SAME write is applied
PROBE["H256"] = PROBE["FT_heavy"]; PROBE["AMORT"] = PROBE["FT_heavy"]
print(f"loaded {len(RUNS)} arms | held-out edits {N} | ef={ef} | K={K}")
print(f"ray zones per sample: target {ZONES.target.sum(1).mean():.1f}, ghost {ZONES.ghost.sum(1).mean():.1f}, "
      f"collateral {ZONES.collateral.sum(1).mean():.1f}, differing {ZONES.differing.sum(1).mean():.1f}")
print("arms with a learned editor network:", list(EDITORNET))

---
## §1 — Did the training work at all? The trained interface

The decisive first check: apply **the exact frozen editor each arm was fine-tuned for** and see whether the edit
lands. The base model is included with the *same* write applied, so this is a like-for-like comparison of one fixed
mechanism across arms.

In [ ]:
# [2] The TRAINED interface: readout injection through each arm's own frozen probe (+ the amortized editor).
@torch.no_grad()
def warm(model):
    st = None
    for t in range(ef): _, st = model.step(obs_t[:, t], st)
    return model.flat_state(st)

@torch.no_grad()
def roll(model, h):
    st = model.state_from_flat(h); out = [model.decode(st)]
    for _ in range(K-1):
        p, st = model.predict_step(st); out.append(p)
    return torch.stack(out, 1).cpu().numpy()

TRAINED, UNST = {}, {}
for r in RUNS:
    m = MODELS[r]; h0 = warm(m)
    UNST[r] = edit_scorecard(roll(m, h0), ZONES, gt_roll)
    if r in EDITORNET:
        with torch.no_grad():
            h_ed = EDITORNET[r](h0, target4)
    else:
        W, b_, Wp = PROBE[r]
        h_ed = readout_inject(h0, target4, W, b_, Wp)
    c = edit_scorecard(roll(m, h_ed), ZONES, gt_roll)
    c["fidelity_ratio"] = fidelity_ratio(c, UNST[r])
    TRAINED[r] = c

rows = ["| arm | trained interface | Edit Index ↑ | Target RMSE ↓ | Ghost RMSE ↓ | Collateral RMSE ↓ | GT-traj RMSE ↓ | fidelity ratio |",
        "|---|---|---|---|---|---|---|---|"]
for r in RUNS:
    c, u = TRAINED[r], UNST[r]
    kind = "amortized `E_θ`" if r in EDITORNET else "frozen-probe readout injection"
    rows.append(f"| {SHORT[r]} | {kind} | **{c['edit_index']:+.2f}** (unsteered {u['edit_index']:+.2f}) | "
                f"{c['target_rmse']:.3f} | {c['ghost_rmse']:.3f} | {c['collateral_rmse']:.3f} | "
                f"{c['gt_traj_rmse']:.3f} | {c['fidelity_ratio']:.2f} |")
display(Markdown("**Table 1 — the trained interface on held-out edits.** Each arm is edited with the *exact* write "
                 "mechanism it was trained for; the base model gets the same write for reference. The unsteered "
                 "value in brackets is that arm's own −1 end.\n\n" + "\n".join(rows)))
for r in RUNS:
    print(f"  {SHORT[r]:<40s} moved {TRAINED[r]['edit_index']-UNST[r]['edit_index']:+.2f} index points from its own unsteered")

---
## §2 — Does it generalise? Mechanism, and content

Two generalisation tests, both of which a genuine "object handle" should pass and a memorised button should fail:

- **Mechanism generalisation** — the standard §4 editors, none of which the model was trained on, computed by
  `scripts/eval_controls.py`. `Readout injection (fresh probe)` is the sharpest of these: the *same mechanism* as
  the trained interface, but with the probe re-fit on the fine-tuned model's own states.
- **Content generalisation** — `FT_heavy_obj0` saw **only object-0 teleports** in training. Splitting the held-out
  set by which object was teleported asks whether it learned "move an object" or "move object 0".

In [ ]:
# [3] Fig 1 — mechanism generalisation: the trained interface vs every untrained write mechanism.
STD = ["Readout injection", "Global-PCA projection", "PCA geodesic", "MLP-probe gradient", "Decoder gradient"]
plt.style.use("default")
fig, ax = plt.subplots(1, 2, figsize=(17, 5.0))
groups = ["trained interface"] + [s + (" (fresh probe)" if s == "Readout injection" else "") for s in STD]
xi = np.arange(len(groups)); w = 0.8/len(RUNS)
for k, r in enumerate(RUNS):
    vals = [TRAINED[r]["edit_index"]] + [EVALJ[r]["editability"][s]["edit_index"] for s in STD]
    ax[0].bar(xi + (k-(len(RUNS)-1)/2)*w, vals, w*0.92, color=COLOR[r], label=SHORT[r])
    base = [UNST[r]["edit_index"]] + [EVALJ[r]["editability"]["Unsteered"]["edit_index"]]*len(STD)
    ax[1].bar(xi + (k-(len(RUNS)-1)/2)*w, np.array(vals)-np.array(base), w*0.92, color=COLOR[r], label=SHORT[r])
for a, ttl, yl in [(ax[0], "(a) Edit Index by write mechanism", "Edit Index (+1 edited … −1 unedited)"),
                   (ax[1], "(b) movement away from that arm's OWN unsteered value\n(isolates the edit from the arm's predictor quality)",
                    "Δ Edit Index vs unsteered")]:
    a.set_xticks(xi); a.set_xticklabels(groups, fontsize=8, rotation=25, ha="right")
    a.set_ylabel(yl); a.set_title(ttl, fontsize=10); a.axhline(0, color="0.3", lw=0.8)
    a.grid(alpha=0.3, axis="y"); style_ax(a)
ax[0].set_ylim(-1.05, 1.05); ax[0].legend(fontsize=7.5, ncol=2, loc="upper left")
fig.suptitle("Fig 1 — does training for ONE editor make the latent editable by OTHERS?", y=1.02, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig1_mechanism_generalisation.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

In [ ]:
# [4] Fig 2 + Table 2 — content generalisation: FT_heavy_obj0 saw ONLY object-0 teleports in training.
def scorecard_subset(model, probe_key, mask, editor_net=None):
    """Re-score the trained interface on a subset of the held-out edits (e.g. only object-1 teleports)."""
    sub = np.where(mask)[0]
    Zs = build_edit_zones(pre_pos=edits.positions[sub, ef-1, :N_OBJ, :].astype(np.float32),
                          tgt_pos=edits.positions[sub, ef, :N_OBJ, :].astype(np.float32),
                          pre_vel=pre_vel[sub], edit_object=oe[sub], sim=sim, n_obj=N_OBJ,
                          traj_pos=edits.positions[sub, ef:ef+K, :N_OBJ, :].astype(np.float32),
                          gt_edited_traj=gt_roll[sub])
    with torch.no_grad():
        st = None
        o = obs_t[sub]
        for t in range(ef): _, st = model.step(o[:, t], st)
        h0 = model.flat_state(st)
        h_ed = editor_net(h0, target4[sub]) if editor_net is not None else \
               readout_inject(h0, target4[sub], *PROBE[probe_key])
    u = edit_scorecard(roll(model, h0), Zs, gt_roll[sub])
    c = edit_scorecard(roll(model, h_ed), Zs, gt_roll[sub])
    c["fidelity_ratio"] = fidelity_ratio(c, u); c["_unsteered"] = u["edit_index"]
    return c

m0, m1 = (oe == 0), (oe == 1)
print(f"held-out split: {m0.sum()} object-0 teleports, {m1.sum()} object-1 teleports")
CG = {}
for r in RUNS:
    net = EDITORNET.get(r)
    CG[r] = {"object 0 (SEEN in training)": scorecard_subset(MODELS[r], r, m0, net),
             "object 1 (UNSEEN by FT_heavy_obj0)": scorecard_subset(MODELS[r], r, m1, net)}

rows = ["| arm | object 0 (trained on) | object 1 (held-out content) | gap |", "|---|---|---|---|"]
for r in RUNS:
    a = CG[r]["object 0 (SEEN in training)"]; b = CG[r]["object 1 (UNSEEN by FT_heavy_obj0)"]
    da = a["edit_index"] - a["_unsteered"]; db = b["edit_index"] - b["_unsteered"]
    rows.append(f"| {SHORT[r]} | {a['edit_index']:+.2f} (Δ{da:+.2f}) | {b['edit_index']:+.2f} (Δ{db:+.2f}) | {db-da:+.2f} |")
display(Markdown("**Table 2 — content generalisation.** Edit Index of the trained interface, split by which object "
                 "was teleported; Δ is movement from that subset's own unsteered value. Only `FT_heavy_obj0` had "
                 "object-1 edits withheld — every other arm saw both, so their gap is the natural difficulty "
                 "difference between the two objects and acts as the control.\n\n" + "\n".join(rows)))

plt.style.use("default")
fig, a = plt.subplots(figsize=(9.5, 4.6))
xi = np.arange(len(RUNS)); w = 0.38
for j, (kk, col) in enumerate([("object 0 (SEEN in training)", "#0072B2"),
                               ("object 1 (UNSEEN by FT_heavy_obj0)", "#D55E00")]):
    a.bar(xi + (j-0.5)*w, [CG[r][kk]["edit_index"] - CG[r][kk]["_unsteered"] for r in RUNS], w,
          color=col, label=kk)
a.set_xticks(xi); a.set_xticklabels([SHORT[r] for r in RUNS], fontsize=8, rotation=25, ha="right")
a.set_ylabel("Δ Edit Index vs that subset's unsteered"); a.axhline(0, color="0.3", lw=0.8)
a.set_title("Fig 2 — content generalisation: does the edit work on the object it was never trained on?", fontsize=11)
a.legend(fontsize=8); a.grid(alpha=0.3, axis="y"); style_ax(a)
fig.tight_layout(); fig.savefig(f"{OUT}/fig2_content_generalisation.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

---
## §3 — Does the edit hold, and what did the training cost?

An edit that lands for one frame and decays is not editability (the decoder-gradient oracle already showed that on
the base model: **+0.94 → −0.12** over 15 steps). And a fine-tune that buys editing by damaging the world model has
not made it editable — it has made it worse.

In [ ]:
# [5] Fig 3 — (a) Edit Index over the rollout for the trained interface; (b) what the training cost.
plt.style.use("default")
fig, ax = plt.subplots(1, 3, figsize=(18.5, 4.6))
sx = np.arange(K)
for r in RUNS:
    ax[0].plot(sx, TRAINED[r]["edit_index_by_step"], color=COLOR[r], lw=2.0, label=SHORT[r])
    ax[0].plot(sx, UNST[r]["edit_index_by_step"], color=COLOR[r], lw=0.9, ls=":", alpha=0.7)
ax[0].plot(sx, EVALJ["H256"]["editability"]["Decoder gradient"]["edit_index_by_step"], color="0.15", lw=1.6, ls="--",
           label="decoder-gradient oracle (base)")
ax[0].axhline(0, color="0.4", ls=":", lw=1.0); ax[0].set_ylim(-1.05, 1.05)
ax[0].set_xlabel("rollout step s (0 = frame ef)"); ax[0].set_ylabel("Edit Index")
ax[0].set_title("(a) does the trained edit HOLD?\n(solid = edited, dotted = that arm's unsteered)", fontsize=10)
ax[0].legend(fontsize=7, loc="upper right"); ax[0].grid(alpha=0.3); style_ax(ax[0])

xi = np.arange(len(RUNS))
ax[1].bar(xi, [EVALJ[r]["nextstep_rmse_vs_clean"] for r in RUNS], 0.6, color=[COLOR[r] for r in RUNS])
ax[1].axhline(EVALJ["H256"]["nextstep_rmse_vs_clean"], color="0.3", ls=":", lw=1.2)
ax[1].annotate("base model", xy=(len(RUNS)-0.4, EVALJ["H256"]["nextstep_rmse_vs_clean"]), fontsize=7.5,
               color="0.35", ha="right", va="bottom")
ax[1].axhline(EVALJ["H256"]["baselines"]["noise_floor_rmse"], color="#D55E00", ls="--", lw=1.2)
ax[1].annotate("observation noise floor", xy=(len(RUNS)-0.4, EVALJ["H256"]["baselines"]["noise_floor_rmse"]),
               fontsize=7.5, color="#D55E00", ha="right", va="bottom")
ax[1].set_xticks(xi); ax[1].set_xticklabels([SHORT[r] for r in RUNS], fontsize=8, rotation=25, ha="right")
ax[1].set_ylabel("next-step RMSE vs clean"); ax[1].set_title("(b) COST: predictive quality after training", fontsize=10)
ax[1].grid(alpha=0.3, axis="y"); style_ax(ax[1])

for key, lab, col in [("pos_r2_linear", "position R² (linear)", "#0072B2"),
                      ("vel_r2_linear", "velocity R² (linear)", "#D55E00"),
                      ("fiber_resid_mlp", "fiber residual (MLP)", "#009E73")]:
    ax[2].plot(xi, [EVALJ[r][key] for r in RUNS], "-o", ms=5, color=col, label=lab)
ax[2].set_xticks(xi); ax[2].set_xticklabels([SHORT[r] for r in RUNS], fontsize=8, rotation=25, ha="right")
ax[2].set_ylabel("score"); ax[2].set_title("(c) COST: what happened to the latent", fontsize=10)
ax[2].legend(fontsize=8); ax[2].grid(alpha=0.3); style_ax(ax[2])
fig.suptitle("Fig 3 — persistence of the trained edit, and the price paid for it", y=1.03, fontsize=12)
fig.tight_layout(); fig.savefig(f"{OUT}/fig3_hold_and_cost.png", dpi=130, bbox_inches="tight")
display(fig); plt.close(fig)

In [ ]:
# [6] Canonical observation-waterfall helper (CLAUDE.md fixed spec): gray on dark, noisy context frames above a
#     dashed edit line, then each column's OWN free-run from step 0 (no shared teacher-forced row).
N_CTX = 6
DARK, TXT, TICK, EDIT_C = "#0a0a14", "#a3adc2", "#808a9d", "#fa8850"
TARGET_C, GHOST_C = "#00E676", "#FF5252"
ctx_obs = edits.obs[:N, ef-N_CTX:ef, :].astype(np.float32)
def _cx(m):
    i = np.where(m)[0]; return i.mean() if i.size else np.nan
tgt_cx = np.array([_cx(ZONES.target[i]) for i in range(N)])
gho_cx = np.array([_cx(ZONES.ghost[i]) for i in range(N)])
SAMPLES = list(np.argsort(ZONES.teleport * (ZONES.ghost.sum(1) >= 3))[::-1][:3])

def waterfall_grid(col_titles, col_bodies, samples, suptitle, fname):
    """col_bodies[c]: (N,K,R) — each column's OWN free-run from step 0 (step 0 decodes frame ef)."""
    ncol = len(col_titles)
    fig, axes = plt.subplots(len(samples), ncol, figsize=(3.0*ncol, 3.4*len(samples)),
                             squeeze=False, facecolor=DARK)
    for r, smp in enumerate(samples):
        for c in range(ncol):
            ax = axes[r][c]; ax.set_facecolor(DARK)
            panel = np.clip(np.concatenate([ctx_obs[smp], col_bodies[c][smp]], 0), 0, 1)
            ax.imshow(panel, aspect="auto", origin="upper", cmap="gray", vmin=0, vmax=1, interpolation="nearest")
            for sp in ax.spines.values(): sp.set_edgecolor(TICK)
            ax.axhline(N_CTX-0.5, color=EDIT_C, lw=1.4, ls="--", alpha=0.95)
            if not np.isnan(tgt_cx[smp]): ax.axvline(tgt_cx[smp], color=TARGET_C, lw=1.6, alpha=0.9)
            if not np.isnan(gho_cx[smp]): ax.axvline(gho_cx[smp], color=GHOST_C, ls="--", lw=1.6, alpha=0.9)
            if r == 0: ax.set_title(col_titles[c], fontsize=8, color=TXT)
            if c == 0:
                ax.set_ylabel(f"sample {smp} (teleport {ZONES.teleport[smp]:.1f})\nsim frame", fontsize=8, color=TXT)
                ax.set_yticks([0, N_CTX, N_CTX+7, N_CTX+14]); ax.set_yticklabels([ef-N_CTX, ef, ef+7, ef+14], fontsize=7)
            else: ax.set_yticks([])
            ax.set_xlabel("ray", fontsize=8, color=TXT); ax.tick_params(colors=TICK, labelsize=7)
    handles = [Line2D([0],[0], color=TARGET_C, lw=2.2, label="object target location"),
               Line2D([0],[0], color=GHOST_C, ls="--", lw=2.2, label="ghost (pre-edit) location"),
               Line2D([0],[0], color=EDIT_C, ls="--", lw=2.2,
                      label=f"edit applied here ({N_CTX} noisy context frames above; every row below is that "
                            f"column's OWN free-run, step 0 = frame {ef})")]
    fig.legend(handles=handles, loc="upper center", ncol=2, fontsize=8.5, frameon=False,
               labelcolor=TXT, bbox_to_anchor=(0.5, 0.955))
    fig.suptitle(suptitle, y=1.0, fontsize=10.5, color=TXT)
    fig.tight_layout(rect=[0, 0, 1, 0.90])
    fig.savefig(f"{OUT}/{fname}", dpi=130, bbox_inches="tight", facecolor=DARK)
    display(fig); plt.close(fig); print("saved", fname)
print("waterfall helper ready; samples:", SAMPLES)

In [ ]:
# [7] Fig 4 — waterfalls: the TRAINED interface on every arm, against GT and the base model's oracle.
@torch.no_grad()
def trained_roll(r):
    m = MODELS[r]; h0 = warm(m)
    h_ed = EDITORNET[r](h0, target4) if r in EDITORNET else readout_inject(h0, target4, *PROBE[r])
    return roll(m, h_ed)
cols = ["GT (sim)", "base: unsteered\n(no edit)"] + [LABEL[r] for r in RUNS] + ["base: decoder\ngradient (ORACLE)"]
bodies = ([gt_roll, np.load(f"{ROOT['H256']}/eval/H256_rollouts.npz")["roll_Unsteered"]]
          + [trained_roll(r) for r in RUNS]
          + [np.load(f"{ROOT['H256']}/eval/H256_rollouts.npz")["roll_Decoder gradient"]])
waterfall_grid(cols, bodies, SAMPLES,
               "Fig 4 — the TRAINED interface on held-out edits: does the object move, and does the old copy clear?",
               "fig4_waterfalls.png")

---
## §4 — Summary

In [ ]:
# [8] Summary table + computed verdict.
rows = ["| arm | trained interface Δ vs own unsteered | best UNTRAINED mechanism Δ | content gap (obj1 − obj0) | next-step RMSE | fidelity ratio |",
        "|---|---|---|---|---|---|"]
for r in RUNS:
    d_tr = TRAINED[r]["edit_index"] - UNST[r]["edit_index"]
    ub = EVALJ[r]["editability"]["Unsteered"]["edit_index"]
    d_un = max(EVALJ[r]["editability"][s]["edit_index"] - ub for s in STD if s != "Decoder gradient")
    a = CG[r]["object 0 (SEEN in training)"]; b = CG[r]["object 1 (UNSEEN by FT_heavy_obj0)"]
    gap = (b["edit_index"]-b["_unsteered"]) - (a["edit_index"]-a["_unsteered"])
    rows.append(f"| {SHORT[r]} | **{d_tr:+.2f}** | {d_un:+.2f} | {gap:+.2f} | "
                f"{EVALJ[r]['nextstep_rmse_vs_clean']:.4f} | {TRAINED[r]['fidelity_ratio']:.2f} |")
display(Markdown("**Table 3 — summary.** All Δ are index points moved from that arm's *own* unsteered value, which "
                 "controls for the fact that a damaged predictor shifts the whole scale.\n\n" + "\n".join(rows)))

print("================ VERDICT (computed, not asserted) ================")
base_d = TRAINED["H256"]["edit_index"] - UNST["H256"]["edit_index"]
for r in RUNS[1:]:
    d_tr = TRAINED[r]["edit_index"] - UNST[r]["edit_index"]
    ub = EVALJ[r]["editability"]["Unsteered"]["edit_index"]
    d_fresh = EVALJ[r]["editability"]["Readout injection"]["edit_index"] - ub
    cost = EVALJ[r]["nextstep_rmse_vs_clean"] - EVALJ["H256"]["nextstep_rmse_vs_clean"]
    print(f"  {SHORT[r]:<40s} trained interface {d_tr:+.2f} (base {base_d:+.2f}) | "
          f"same mechanism, fresh probe {d_fresh:+.2f} | prediction cost {cost:+.4f}")
best = max(RUNS[1:], key=lambda r: TRAINED[r]["edit_index"] - UNST[r]["edit_index"])
d_best = TRAINED[best]["edit_index"] - UNST[best]["edit_index"]
print(f"\n1. DOES TRAINING MOVE THE EDIT? best arm = {SHORT[best]}, {d_best:+.2f} index points vs the base model's "
      f"{base_d:+.2f} on the identical write.")
print(f"   Absolute level: {TRAINED[best]['edit_index']:+.2f} on a scale where its own unsteered is "
      f"{UNST[best]['edit_index']:+.2f} and the decoder-gradient oracle reaches "
      f"{EVALJ['H256']['editability']['Decoder gradient']['edit_index']:+.2f}.")
gen = {r: max(EVALJ[r]["editability"][s]["edit_index"] - EVALJ[r]["editability"]["Unsteered"]["edit_index"]
              for s in STD if s != "Decoder gradient") for r in RUNS}
print(f"2. MECHANISM GENERALISATION: best UNTRAINED write mechanism moves "
      f"{gen[best]:+.2f} on {SHORT[best]} vs {gen['H256']:+.2f} on the base model -> "
      + ("training transfers across write mechanisms" if gen[best] > gen["H256"] + 0.15
         else "NO transfer: the model obeys the interface it was trained for, not the others"))
a = CG["FT_heavy_obj0"]["object 0 (SEEN in training)"]; b = CG["FT_heavy_obj0"]["object 1 (UNSEEN by FT_heavy_obj0)"]
ga = (b["edit_index"]-b["_unsteered"]) - (a["edit_index"]-a["_unsteered"])
ctrl = CG["FT_heavy"]["object 1 (UNSEEN by FT_heavy_obj0)"]; ctrl0 = CG["FT_heavy"]["object 0 (SEEN in training)"]
gc = (ctrl["edit_index"]-ctrl["_unsteered"]) - (ctrl0["edit_index"]-ctrl0["_unsteered"])
print(f"3. CONTENT GENERALISATION: object-0-only arm has an obj1−obj0 gap of {ga:+.2f}; the both-objects control "
      f"(FT_heavy) has {gc:+.2f} -> "
      + ("withholding an object HURTS it -> content-specific, a button per object" if ga < gc - 0.1
         else "no content-specific penalty beyond the natural object difficulty difference"))
print(f"4. COST: next-step RMSE base {EVALJ['H256']['nextstep_rmse_vs_clean']:.4f} -> "
      + ", ".join(f"{r} {EVALJ[r]['nextstep_rmse_vs_clean']:.4f}" for r in RUNS[1:]))
print(f"   Retention matters: FT_heavy {EVALJ['FT_heavy']['nextstep_rmse_vs_clean']:.4f} vs FT_heavy_noret "
      f"{EVALJ['FT_heavy_noret']['nextstep_rmse_vs_clean']:.4f} (noise floor "
      f"{EVALJ['H256']['baselines']['noise_floor_rmse']:.4f}).")
print("\nPNGs:", sorted(os.listdir(OUT)))